In [ ]:
!pip install -q langgraph groq grandalf

In [ ]:
import os
import json
import importlib.util
import re
from google.colab import userdata
import pandas as pd
import smtplib
from email.mime.text import MIMEText
from groq import Groq
from langgraph.graph import StateGraph, END
from typing import TypedDict
import importlib.util
import grandalf

In [ ]:


import os
from groq import Groq


import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"


client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print("Key loaded:", os.environ.get("GROQ_API_KEY")[:8], "...")



Key loaded: gsk_t6Wa ...


In [ ]:
# =========================================================
#  CSV UPLOAD
# =========================================================
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
print("\nUploaded Data Preview:")
print(df.head())

data = df.to_dict(orient="records")

Saving Sale.csv to Sale.csv

Uploaded Data Preview:
   Year  Sale
0  2017   110
1  2018   120
2  2019   180
3  2020   200
4  2021   180


In [ ]:
# =========================================================
# CREATE SKILLS
# =========================================================
import os
os.makedirs("skills/schema_analyzer", exist_ok=True)
os.makedirs("skills/data_analysis", exist_ok=True)
os.makedirs("skills/insight_generation", exist_ok=True)

# -------- SCHEMA ANALYZER --------
with open("skills/schema_analyzer/SKILL.md", "w") as f:
    f.write("""name: schema_analyzer
description: Understand dataset schema and identify important columns
""")

with open("skills/schema_analyzer/skill.py", "w") as f:
    f.write(f"""
from groq import Groq
import os
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def run(state):
    print("[Skill] Schema Analyzer")

    sample = state["data"][:5]

    prompt = f\"\"\"
Analyze dataset schema:

Sample:
{{sample}}

Identify:
- time column
- numeric columns
- important features
\"\"\"

    res = client.chat.completions.create(
        model="{MODEL}",
        messages=[{{"role": "user", "content": prompt}}],
        temperature=0
    )

    state["schema"] = res.choices[0].message.content
    return state
""")

# -------- DATA ANALYSIS --------
with open("skills/data_analysis/SKILL.md", "w") as f:
    f.write("""name: data_analysis
description: Perform analysis on numeric columns
""")

with open("skills/data_analysis/skill.py", "w") as f:
    f.write("""
def run(state):
    print("[Skill] Data Analysis")

    data = state["data"]
    sample = data[0]

    numeric_cols = [k for k,v in sample.items() if isinstance(v,(int,float))]

    if not numeric_cols:
        state["analysis"] = "No numeric columns found"
        return state

    col = numeric_cols[0]

    values = [row[col] for row in data]

    growth = []
    for i in range(1,len(values)):
        growth.append(round((values[i]-values[i-1])/values[i-1]*100,2))

    state["analysis"] = {
        "column_used": col,
        "growth_rates": growth
    }

    return state
""")

# -------- INSIGHT GENERATION --------
with open("skills/insight_generation/SKILL.md", "w") as f:
    f.write("""name: insight_generation
description: Generate business insights from analysis
""")

with open("skills/insight_generation/skill.py", "w") as f:
    f.write(f"""
from groq import Groq
import os
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def run(state):
    print("[Skill] Insight Generation")

    prompt = f\"\"\"
Generate 3 business insights:

{{state['analysis']}}
\"\"\"

    res = client.chat.completions.create(
        model="{MODEL}",
        messages=[{{"role": "user", "content": prompt}}],
        temperature=0.3
    )

    state["insights"] = res.choices[0].message.content
    return state
""")

In [ ]:

# =========================================================
# LOAD SKILLS
# =========================================================
def load_skills():
    skills = {}

    for folder in os.listdir("skills"):
        path = f"skills/{folder}/SKILL.md"
        if not os.path.isfile(path):
            continue

        name, desc = None, None

        with open(path) as f:
            for line in f:
                line = line.strip()
                if line.startswith("name:"):
                    name = line.split("name:")[1].strip()
                elif line.startswith("description:"):
                    desc = line.split("description:")[1].strip()

        if name and desc:
            skills[name] = {"folder": folder, "description": desc}

    return skills


def run_skill(name, state, skills):
    path = f"skills/{skills[name]['folder']}/skill.py"
    spec = importlib.util.spec_from_file_location("skill", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.run(state)


In [ ]:
# =========================================================
# PLAN CLEANER
# =========================================================
def clean_plan(text, skills):
    text = text.replace("```", "").strip()
    skill_names = list(skills.keys())

    steps = []
    for skill in skill_names:
        if skill in text:
            steps.append(skill)

    return steps

In [ ]:
# =========================================================
# STATE
# =========================================================
class State(TypedDict):
    query: str
    data: list
    schema: str
    analysis: dict
    insights: str
    plan: list
    step: int

In [ ]:
# =========================================================
# PLANNER
# =========================================================
def planner(state):
    skills = load_skills()

    desc = "\n".join([f"{k}:{v['description']}" for k,v in skills.items()])

    prompt = f"""
You are an AI agent.

Available skills:
{desc}

User query:
{state['query']}

Return ONLY:
schema_analyzer -> data_analysis -> insight_generation

No explanation. No markdown.
"""

    res = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    raw = res.choices[0].message.content.strip()
    print("[RAW PLAN]:", raw)

    steps = clean_plan(raw, skills)
    print("[CLEAN PLAN]:", steps)

    return {"plan": steps, "step": 0}

In [ ]:
# =========================================================
# ROUTER
# =========================================================
def router(state):
    if state["step"] >= len(state["plan"]):
        return END
    return "executor"

In [ ]:
# =========================================================
# EXECUTOR
# =========================================================
def executor(state):
    skills = load_skills()
    step = state["plan"][state["step"]]

    print(f"[Executing]: {step}")
    return run_skill(step, state, skills)

def increment(state):
    return {"step": state["step"] + 1}

In [ ]:
# =========================================================
# GRAPH
# =========================================================
graph = StateGraph(State)

graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("increment", increment)

graph.set_entry_point("planner")
graph.add_conditional_edges("planner", router)
graph.add_conditional_edges("increment", router)
graph.add_edge("executor", "increment")

app = graph.compile()

In [ ]:

# =========================================================
# RUN DEMO
# =========================================================
state = {
    "query": "Analyze dataset and generate insights",
    "data": data,
    "schema": "",
    "analysis": {},
    "insights": "",
    "plan": [],
    "step": 0
}

print("\n=== EXECUTION FLOW ===")
for e in app.stream(state):
    for k in e:
        print("[Node]:", k)

res = app.invoke(state)

print("\n=========== FINAL INSIGHTS ===========")
print(res["insights"])


=== EXECUTION FLOW ===
[RAW PLAN]: schema_analyzer
Identified important columns: id, name, age, salary, department
Data types: id(int), name(str), age(int), salary(float), department(str)

data_analysis
Correlation between age and salary: 0.7
Average salary by department: 
- Sales: 60000
- Marketing: 55000
- IT: 70000

insight_generation
Employees in the IT department tend to earn higher salaries compared to other departments.
There is a strong positive correlation between age and salary, suggesting that older employees are likely to earn higher salaries.
Consider implementing a salary increase for employees in the IT department to improve employee retention and satisfaction.
[CLEAN PLAN]: ['schema_analyzer', 'data_analysis', 'insight_generation']
[Node]: planner
[Executing]: schema_analyzer
[Skill] Schema Analyzer
[Node]: executor
[Node]: increment
[Executing]: data_analysis
[Skill] Data Analysis
[Node]: executor
[Node]: increment
[Executing]: insight_generation
[Skill] Insight Gener

In [ ]:
#  SET YOUR GROQ API KEY
# =========================================================
client = Groq(api_key="YOUR_GROQ_API_KEY")
MODEL = "llama-3.1-8b-instant"

In [ ]:
# =========================================================
# 1. CREATE SKILL STRUCTURE
# =========================================================
os.makedirs("skills/data_analysis", exist_ok=True)
os.makedirs("skills/insight_generation", exist_ok=True)
os.makedirs("skills/email_formatter", exist_ok=True)

# -------- DATA ANALYSIS --------
with open("skills/data_analysis/SKILL.md", "w") as f:
    f.write("""name: data_analysis
description: Analyze structured data and compute growth trends
""")

with open("skills/data_analysis/skill.py", "w") as f:
    f.write("""
def run(state):
    print("[Skill] Data Analysis")

    data = state["data"]
    years = list(data.keys())
    growth = []

    for i in range(1, len(years)):
        prev = data[years[i-1]]
        curr = data[years[i]]
        growth.append(round((curr - prev) / prev * 100, 2))

    state["analysis"] = {
        "growth_rates": growth,
        "values": data
    }

    return state
""")

# -------- INSIGHT GENERATION --------
with open("skills/insight_generation/SKILL.md", "w") as f:
    f.write("""name: insight_generation
description: Generate business insights from analyzed data
""")

with open("skills/insight_generation/skill.py", "w") as f:
    f.write(f"""
from groq import Groq
import os

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def run(state):
    print("[Skill] Insight Generation (LLM)")

    prompt = f\"\"\"
Generate 3 business insights:

{{state['analysis']}}
\"\"\"

    response = client.chat.completions.create(
        model="{MODEL}",
        messages=[{{"role": "user", "content": prompt}}],
        temperature=0.3
    )

    state["insights"] = response.choices[0].message.content
    return state
""")

# -------- EMAIL FORMATTER --------
with open("skills/email_formatter/SKILL.md", "w") as f:
    f.write("""name: email_formatter
description: Convert insights into executive email
""")

with open("skills/email_formatter/skill.py", "w") as f:
    f.write("""
def run(state):
    print("[Skill] Email Formatter")

    state["final_output"] = f'''
Subject: Sales Insights Report

Dear Leadership,

Key Insights:
{state["insights"]}

Regards,
AI Data Analyst
'''
    return state
""")


In [ ]:
# =========================================================
# 2. LOAD SKILLS (FROM SKILL.md)
# =========================================================
def load_skills():
    skills = {}
    base = "skills"

    for folder in os.listdir(base):
        path = os.path.join(base, folder)

        if os.path.isdir(path):
            with open(os.path.join(path, "SKILL.md")) as f:
                lines = f.read().split("\n")

            name = lines[0].replace("name:", "").strip()
            desc = lines[1].replace("description:", "").strip()

            skills[name] = {
                "folder": folder,
                "description": desc
            }

    return skills

In [ ]:
# =========================================================
# 3. EXECUTE SKILL (DYNAMIC LOADER)
# =========================================================
def execute_skill(skill_name, state, skills):
    folder = skills[skill_name]["folder"]
    file_path = f"skills/{folder}/skill.py"

    spec = importlib.util.spec_from_file_location("skill", file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module.run(state)


In [ ]:
# =========================================================
# 4. STATE
# =========================================================
class AgentState(TypedDict):
    query: str
    data: dict
    analysis: dict
    insights: str
    final_output: str
    plan: list
    step: int


In [ ]:
# =========================================================
# 5. LLM PLANNER
# =========================================================
def planner(state: AgentState):
    print("\n[LLM] Planning...")

    skills = load_skills()

    skill_desc = "\n".join(
        [f"{k}: {v['description']}" for k, v in skills.items()]
    )

    prompt = f"""
Available skills:
{skill_desc}

User query:
{state['query']}

Return:
data_analysis -> insight_generation -> email_formatter

Do NOT explain.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    plan_text = response.choices[0].message.content.strip()
    print("[PLAN]:", plan_text)

    return {
        "plan": [s.strip() for s in plan_text.split("->")],
        "step": 0
    }


In [ ]:
# =========================================================
# 6. ROUTER
# =========================================================
def router(state: AgentState):
    if state["step"] >= len(state["plan"]):
        return END
    return "executor"


In [ ]:
# =========================================================
# 7. EXECUTOR NODE
# =========================================================
def executor(state: AgentState):
    skills = load_skills()
    step = state["plan"][state["step"]]

    print(f"\n[Executing]: {step}")

    return execute_skill(step, state, skills)

def increment(state: AgentState):
    return {"step": state["step"] + 1}


In [ ]:
# =========================================================
# 8. BUILD GRAPH
# =========================================================
graph = StateGraph(AgentState)

graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("increment", increment)

graph.set_entry_point("planner")

graph.add_conditional_edges("planner", router)
graph.add_conditional_edges("increment", router)

graph.add_edge("executor", "increment")

app = graph.compile()

In [ ]:

# =========================================================
# 9. RUN DEMO
# =========================================================
data = {
    "2021": 100,
    "2022": 110,
    "2023": 120,
    "2024": 150
}

input_state = {
    "query": "Analyze sales data and send insights via email",
    "data": data,
    "analysis": {},
    "insights": "",
    "final_output": "",
    "plan": [],
    "step": 0
}

# 🔥 PRINT EXECUTION FLOW (BEST FOR DEMO)
print("\n=== EXECUTION FLOW ===")
for event in app.stream(input_state):
    for node in event:
        print(f"[Node]: {node}")

# FINAL OUTPUT
result = app.invoke(input_state)

print("\n=========== FINAL OUTPUT ===========")
print(result["final_output"])


=== EXECUTION FLOW ===

[LLM] Planning...
[PLAN]: data_analysis -> insight_generation -> email_formatter
[Node]: planner

[Executing]: data_analysis
[Skill] Data Analysis
[Node]: executor
[Node]: increment

[Executing]: insight_generation
[Skill] Insight Generation (LLM)
[Node]: executor
[Node]: increment

[Executing]: email_formatter
[Skill] Email Formatter
[Node]: executor
[Node]: increment

[LLM] Planning...
[PLAN]: data_analysis -> insight_generation -> email_formatter

[Executing]: data_analysis
[Skill] Data Analysis

[Executing]: insight_generation
[Skill] Insight Generation (LLM)

[Executing]: email_formatter
[Skill] Email Formatter

=========== FINAL OUTPUT ===========

Subject: Sales Insights Report

Dear Leadership,

Key Insights:
Based on the provided data, here are three business insights:

1. **Consistent Growth in Revenue**: The company has shown a consistent growth in revenue over the years, with an average annual growth rate of 12.36% (calculated by taking the average 